# Notebook 1: Repository-Level Evaluation

**Goal:** Explain what happened for a single target repository.

**Inputs:** `evaluation_attempts.csv`, `evaluation_candidates.csv`, `generated_tests.csv`, `failure_cases.csv`

**RQs addressed:** RQ1, RQ2, RQ3, RQ7, RQ8, RQ9

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.normalize import build_candidate_summary
from analysis.plots import (
    outcome_funnel,
    stacked_outcome_bar,
    metric_delta_distribution,
    before_after_slope,
    cost_vs_improvement_scatter,
    failure_category_bar,
)

sns.set_theme(style='whitegrid')

DATA_DIR = Path('../output')
attempts   = pd.read_csv(DATA_DIR / 'evaluation_attempts.csv')
candidates = pd.read_csv(DATA_DIR / 'evaluation_candidates.csv')
gen_tests  = pd.read_csv(DATA_DIR / 'generated_tests.csv') if (DATA_DIR / 'generated_tests.csv').exists() else pd.DataFrame()
failures   = pd.read_csv(DATA_DIR / 'failure_cases.csv')   if (DATA_DIR / 'failure_cases.csv').exists()   else pd.DataFrame()

In [ ]:
# ── Select repository ──────────────────────────────────────────────────────
REPO_OWNER  = 'consulthunter'
REPO_NAME   = 'TestMap-Example'
COMMIT_HASH = None   # None = any commit

mask = (attempts['repo_owner'] == REPO_OWNER) & (attempts['repo_name'] == REPO_NAME)
if COMMIT_HASH:
    mask &= attempts['commit_hash'] == COMMIT_HASH

repo_attempts   = attempts[mask].copy()
repo_candidates = candidates[candidates['repo_name'] == REPO_NAME].copy()
print(f'Attempts in repo: {len(repo_attempts)}')
print(f'Candidates:       {len(repo_candidates)}')

## 1. Repository Scope and Baseline

In [ ]:
# Baseline coverage and mutation scores, candidate count, project count
print(repo_attempts[['repo_owner', 'repo_name', 'commit_hash']].drop_duplicates())
print('\nCoverage before (median):', repo_attempts['coverage_before'].median())
print('Mutation score before (median):', repo_attempts['mutation_score_before'].median())

## 2. Lane Outcome Funnels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
outcome_funnel(repo_attempts, lane='llm',     ax=axes[0])
outcome_funnel(repo_attempts, lane='agentic', ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Candidate-Level Results

In [ ]:
# Candidate outcome table: candidate_key, lane, any_validated_success, best_coverage_delta, attempt_count
display_cols = [c for c in [
    'candidate_key', 'lane', 'any_validated_success',
    'best_coverage_delta', 'best_mutation_delta', 'attempt_count'
] if c in repo_candidates.columns]
repo_candidates[display_cols].sort_values('any_validated_success', ascending=False).head(20)

## 4. Generated Test Volume and Quality

In [ ]:
if not gen_tests.empty:
    repo_tests = gen_tests[gen_tests.get('repo_name', pd.Series()) == REPO_NAME]
    print(f'Generated tests for this repo: {len(repo_tests)}')
    display(repo_tests.head(10))
else:
    print('No generated_tests.csv available.')

## 5. Coverage and Mutation Movement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
before_after_slope(repo_attempts, 'coverage_before', 'coverage_after', ax=axes[0])
before_after_slope(repo_attempts, 'mutation_score_before', 'mutation_score_after', ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Cost and Runtime

In [ ]:
if 'duration_seconds' in repo_attempts.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    cost_vs_improvement_scatter(repo_attempts, 'duration_seconds', 'coverage_delta', 'lane', ax=ax)
    plt.show()

print(repo_attempts.groupby('lane')[['duration_seconds', 'total_tokens']].agg(['median', 'sum']))

## 7. Agentic Change Footprint

In [ ]:
agentic = repo_attempts[repo_attempts['lane'] == 'agentic']
footprint_cols = [c for c in [
    'changed_files_count', 'test_files_changed',
    'production_files_changed', 'project_files_changed', 'deleted_files_count'
] if c in agentic.columns]
if footprint_cols:
    agentic[footprint_cols].describe().T

## 8. LLM Failure Taxonomy and Roslyn Diagnostics

In [ ]:
llm = repo_attempts[repo_attempts['lane'] == 'llm']
fig, ax = plt.subplots(figsize=(10, 4))
failure_category_bar(llm, lane_col='lane', category_col='failure_category', ax=ax)
plt.show()

## 9. Agentic Failure Outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
stacked_outcome_bar(agentic, group_col='tool_id', outcome_col='observed_outcome', ax=ax)
plt.show()

## 10. Qualitative Failure Case Sample

In [ ]:
if not failures.empty:
    repo_failures = failures[failures.get('repo_name', pd.Series()) == REPO_NAME]
    cols = [c for c in ['failure_case_id', 'lane', 'preliminary_failure_label', 'source_method_name', 'outcome_summary'] if c in repo_failures.columns]
    repo_failures[cols].head(10)
else:
    print('No failure_cases.csv available. Run: python -m analysis export-failures ...')

## 11. Repository Summary

In [ ]:
from analysis.summaries import build_overview
import json

overview = build_overview(repo_attempts)
print(json.dumps(overview, indent=2, default=str))